# EDA — GlobalLandTemperaturesByCity (raw)

Eksploracyjna analiza surowego datasetu przed przetworzeniem w pipeline Kedro.

**Plik:** `data/01_raw/GlobalLandTemperaturesByCity.csv`

In [1]:
import csv
import sys
import os
from pathlib import Path

DATA_PATH = Path(r'c:\DEVELOPING\STUDIA\ASI\Projekt\new-kedro-project\data\01_raw\GlobalLandTemperaturesByCity.csv')

## 1. Podstawowe statystyki — nulle, unikalność, zakres dat

In [2]:
total = 0
null_temp = 0
null_uncertainty = 0
cities = set()
countries = set()
min_date = None
max_date = None

with open(DATA_PATH, encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        total += 1
        if not row['AverageTemperature']:
            null_temp += 1
        if not row['AverageTemperatureUncertainty']:
            null_uncertainty += 1
        cities.add(row['City'])
        countries.add(row['Country'])
        d = row['dt']
        if min_date is None or d < min_date:
            min_date = d
        if max_date is None or d > max_date:
            max_date = d

print(f'Total rows:                          {total}')
print(f'Null AverageTemperature:             {null_temp} ({null_temp/total*100:.1f}%)')
print(f'Null AverageTemperatureUncertainty:  {null_uncertainty} ({null_uncertainty/total*100:.1f}%)')
print(f'Unique cities:                       {len(cities)}')
print(f'Unique countries:                    {len(countries)}')
print(f'Date range:                          {min_date} to {max_date}')

Total rows:                          8599212
Null AverageTemperature:             364130 (4.2%)
Null AverageTemperatureUncertainty:  364130 (4.2%)
Unique cities:                       3448
Unique countries:                    159
Date range:                          1743-11-01 to 2013-09-01


## 2. Format współrzędnych geograficznych i duplikaty

In [3]:
lat_formats = set()
lon_formats = set()
seen = {}
dups = 0

with open(DATA_PATH, encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['Latitude']:
            lat_formats.add(row['Latitude'][-1])
        if row['Longitude']:
            lon_formats.add(row['Longitude'][-1])
        key = (row['dt'], row['City'], row['Country'])
        if key in seen:
            dups += 1
        else:
            seen[key] = 1

print('Latitude suffix chars:', lat_formats)
print('Longitude suffix chars:', lon_formats)
print('Duplicate (dt, City, Country) combos:', dups)

Latitude suffix chars: {'N', 'S'}
Longitude suffix chars: {'W', 'E'}
Duplicate (dt, City, Country) combos: 46034


## 3. Rozkład nulli w czasie — czy stare dane są gorsze?

In [4]:
null_by_period = {}
total_by_period = {}

with open(DATA_PATH, encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        year = int(row['dt'][:4])
        period = (year // 50) * 50
        total_by_period[period] = total_by_period.get(period, 0) + 1
        if not row['AverageTemperature']:
            null_by_period[period] = null_by_period.get(period, 0) + 1

print('Null rate by 50-year period:')
for period in sorted(total_by_period):
    n = null_by_period.get(period, 0)
    t = total_by_period[period]
    bar = '#' * int(n / t * 40)
    print(f'  {period}-{period+49}: {n:>6}/{t:<6} = {n/t*100:5.1f}%  {bar}')

Null rate by 50-year period:
  1700-1749:  43066/52244  =  82.4%  ################################
  1750-1799:  32538/508661 =   6.4%  ##
  1800-1849: 161568/1176546 =  13.7%  #####
  1850-1899: 123888/2070611 =   6.0%  ##
  1900-1949:      0/2106000 =   0.0%  
  1950-1999:      0/2106000 =   0.0%  
  2000-2049:   3070/579150 =   0.5%  


## 4. Zakres temperatur i outliers

In [5]:
temps = []
with open(DATA_PATH, encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['AverageTemperature']:
            temps.append(float(row['AverageTemperature']))

temps.sort()
n = len(temps)

print(f'Non-null temps:  {n}')
print(f'Min:             {temps[0]}')
print(f'Max:             {temps[-1]}')
print(f'P1:              {temps[int(n*0.01)]}')
print(f'P99:             {temps[int(n*0.99)]}')
print(f'P0.1:            {temps[int(n*0.001)]}')
print(f'P99.9:           {temps[int(n*0.999)]}')

extreme = [t for t in temps if t < -60 or t > 50]
print(f'\nExtreme values (< -60 or > 50): {len(extreme)}')
if extreme:
    print('Sample extremes:', extreme[:10])

Non-null temps:  8235082
Min:             -42.70399999999999
Max:             39.651
P1:              -13.343
P99:             32.546
P0.1:            -22.974
P99.9:           35.27500000000001

Extreme values (< -60 or > 50): 0
